In [ ]:
from uq_physicell.model_analysis import ModelAnalysisContext, run_simulations

# Run a single simulation to generate observational data
db_path = "ex7_ObsData.db"  # Path to the database file
model_config = {"ini_path": "uq_pc_struc.ini", "struc_name": "Model_struc"} # Example model configuration
sampler = 'Custom'  # Example sampler
params_info = { # Example parameters information
    'mac_phag_rate_infected': {'ref_value': 0.3, 'perturbation': None},
    'mac_motility_speed': {'ref_value': 1.0, 'perturbation': None},
    'epi2infected_sat': {'ref_value': 0.1, 'perturbation': None},
    'epi2infected_hfm': {'ref_value': 0.18, 'perturbation': None}
}  
qoi_funcs = {
    "epithelial_count": lambda df_cell: len(df_cell[df_cell['cell_type'] == 'epithelial']),
    "infected_count":   lambda df_cell: len(df_cell[df_cell['cell_type'] == 'epithelial_infected']),
}
# Create Model Analysis Context
context = ModelAnalysisContext(db_path, model_config, sampler, params_info, qoi_funcs, num_workers=8)
# Sample parameter sets and run simulations
context.dic_samples = {0: {'mac_phag_rate_infected': 0.3, 'mac_motility_speed': 1.0, 'epi2infected_sat': 0.1, 'epi2infected_hfm': 0.18}}
run_simulations(context)

Inserting parameter: mac_phag_rate_infected with properties: {'ref_value': 0.3, 'perturbation': None}
Inserting parameter: mac_motility_speed with properties: {'ref_value': 1.0, 'perturbation': None}
Inserting parameter: epi2infected_sat with properties: {'ref_value': 0.1, 'perturbation': None}
Inserting parameter: epi2infected_hfm with properties: {'ref_value': 0.18, 'perturbation': None}
Inserting {'epithelial_count': "lambda df_cell: len(df_cell[df_cell['cell_type'] == 'epithelial'])", 'infected_count': "lambda df_cell: len(df_cell[df_cell['cell_type'] == 'epithelial_infected'])"} QoIs into the database
Simulations completed and results stored in the database: Ex7_ObsData.db.


In [2]:
import pandas as pd
import numpy as np
from uq_physicell.database.ma_db import load_output
# Load the simulation results and extract cell counts over time
df_qois_data = load_output(db_path)
df_all_data = pd.DataFrame()
for index, row in df_qois_data.iterrows():
    df_temp = row['Data']
    df_all_data = pd.concat([df_all_data, df_temp], ignore_index=True)

# Aggregate with multiple summary statistics
df_aggregated = df_all_data.groupby('time').agg({
    'epithelial_count': ['mean', 'std', 'min', 'max', 'count'],
    'infected_count': ['mean', 'std', 'min', 'max', 'count']
}).reset_index()

# Flatten column names for easier access
df_aggregated.columns = ['_'.join(col).strip('_') if col[1] else col[0] 
                          for col in df_aggregated.columns.values]

time = df_aggregated['time'].values
live = df_aggregated['epithelial_count_mean'].values
dead = df_aggregated['infected_count_mean'].values
df_aggregated.rename(columns={'time': 'Time', 'epithelial_count_mean': 'Healthy Epithelial Cells', 'infected_count_mean': 'Infected Epithelial Cells'}, inplace=True)
display(df_aggregated)
# Add noise to the aggregated to columns Healthy Epithelial Cells and Infected Epithelial Cells, and save to CSV
columns = ['Healthy Epithelial Cells', 'Infected Epithelial Cells']
noise_std = 0.05  # Standard deviation of the noise (Normal distribution) no negative values
for col in columns:
    df_aggregated[col] = np.maximum(0, df_aggregated[col] + df_aggregated[col] * noise_std * np.random.randn(len(df_aggregated)))
df_aggregated.to_csv("ex7_ObsData.csv", index=False, columns=['Time']+columns)

,Time,Healthy Epithelial Cells,epithelial_count_std,epithelial_count_min,epithelial_count_max,epithelial_count_count,Infected Epithelial Cells,infected_count_std,infected_count_min,infected_count_max,infected_count_count
0,0.0,250.0,0.000000,250,250,10,20.0,0.000000,20,20,10
1,30.0,240.3,2.311805,237,243,10,27.6,2.270585,25,31,10
2,60.0,204.3,7.288499,196,218,10,57.8,7.495184,44,66,10
3,90.0,143.4,13.993649,125,168,10,114.2,12.568037,94,131,10
4,120.0,83.3,17.179122,55,104,10,175.6,16.084499,158,201,10
5,150.0,46.3,15.405987,19,60,10,212.6,15.791700,196,238,10
6,180.0,18.1,10.660936,0,33,10,239.9,11.532081,226,259,10
7,210.0,3.6,4.221637,0,11,10,258.3,5.677441,247,266,10
8,240.0,0.2,0.632456,0,2,10,263.9,5.820462,254,272,10
9,270.0,0.1,0.316228,0,1,10,267.9,7.093816,258,279,10
